# Phase 5 — 3D verification of the top Pareto designs

**Runs in parallel with the Phase-4 campaign** — on a *separate* Colab runtime. It reads the campaign's Google-Drive checkpoint, picks the current top-k structurally-diverse designs, and re-evaluates each with a **3D unsteady SU2** run (single V-unit blade, external flow) to check the cheap 2D-slice ranking survives the 3D physics (finite span / tip).

Thin orchestration — logic lives in tested `fanopt.cfd.{mesh,phase5}` + `scripts/run_phase5_verify.py`. **CPU only** (SU2; no GPU — that's PyFR, a separate step). The 3D unsteady runs are the cost; designs run in parallel processes.


## 1. Repo + Python/native deps


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Verification runs each design's CFD in its own PROCESS; keep each single-threaded
# so N_WORKERS processes use N_WORKERS cores cleanly (no BLAS/SU2 oversubscription).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    REPO_DIR = Path("/content/fan-optimization")
    BRANCH = "main"
    REPO_URL = "https://github.com/clingergab/fan-optimization.git"
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
    subprocess.run(
        "apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
        "libxft2 libxinerama1 unzip".split(),
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "gmsh", "cadquery", "scipy", "jinja2", "pyyaml", "botorch", "tqdm", "ipywidgets"],
        check=True,
    )
else:
    REPO_DIR = Path.cwd()
    while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / "pyproject.toml").exists():
        REPO_DIR = REPO_DIR.parent

for p in (REPO_DIR / "src", REPO_DIR / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

assert (REPO_DIR / "scripts" / "run_phase5_verify.py").exists(), \
    "phase-5 code missing — is the branch pushed?"

_missing = [m for m in ("cadquery", "gmsh", "botorch", "scipy", "yaml")
            if importlib.util.find_spec(m) is None]
if _missing:
    raise ModuleNotFoundError(
        f"Missing {_missing} in this kernel:\n  {sys.executable}\n"
        "Phase 5 needs cadquery + gmsh + botorch. Local: pick the kernel that has them "
        "(e.g. 'fanopt-local'); Colab: re-run this cell (it installs them)."
    )
print("repo:", REPO_DIR, "| colab:", IN_COLAB)


## 2. SU2 — install (Colab) or locate (local)


In [ ]:
import importlib.util
import os
import subprocess
import urllib.request
from pathlib import Path

if importlib.util.find_spec("google.colab") is not None:
    SU2_VERSION = "8.0.1"
    SU2_DIR = Path("/content/su2")
    if not any(SU2_DIR.rglob("SU2_CFD")):
        url = (
            f"https://github.com/su2code/SU2/releases/download/"
            f"v{SU2_VERSION}/SU2-v{SU2_VERSION}-linux64.zip"
        )
        print("[su2] downloading", url)
        urllib.request.urlretrieve(url, "/tmp/su2.zip")
        SU2_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/su2.zip", "-d", str(SU2_DIR)], check=True)
    cands = [p for p in SU2_DIR.rglob("SU2_CFD") if p.is_file() and not p.is_symlink()]
    assert cands, "SU2_CFD not found after extract"
    bindir = cands[0].parent
    for p in bindir.iterdir():
        if p.is_file():
            os.chmod(p, 0o755)
else:
    bindir = Path.home() / "su2-local" / "extracted" / "bin"
    assert (bindir / "SU2_CFD").exists(), f"SU2_CFD not at {bindir}"

os.environ["SU2_RUN"] = str(bindir)
os.environ["PATH"] = str(bindir) + os.pathsep + os.environ.get("PATH", "")
print("SU2_RUN:", os.environ["SU2_RUN"])


## 3. Settings + campaign source (Drive)


In [ ]:
from pathlib import Path

# --- What to verify + fidelity ---------------------------------------------
TOP_K = 3               # top-k structurally-diverse Pareto designs
N_WORKERS = 3           # designs run in parallel PROCESSES; useful ≈ min(TOP_K, cores)
# 3D unsteady fidelity — higher = more accurate but much slower per design:
N_CYCLES = 3            # oscillation cycles (cycle 1 discarded); production ranking = 5
INNER_ITER = 30         # dual-time inner iterations per step
WALL_MESH_SIZE_M = 0.002  # blade-surface element size (coarsen for a quicker check)

# --- Read the (possibly still-running) campaign, write verification ---------
# Safe to run on a SEPARATE Colab runtime while the Phase-4 campaign runs: it
# reads the campaign's Drive checkpoint (the current top-k) — re-run to refresh
# as the campaign sharpens the front. Uses its own cores (don't share a runtime).
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CAMPAIGN_DIR = Path("/content/drive/MyDrive/fanopt/phase4_bo")
    OUT_DIR = Path("/content/drive/MyDrive/fanopt/phase5_verify")
else:
    CAMPAIGN_DIR = REPO_DIR / "data" / "phase4_bo_nb"
    OUT_DIR = REPO_DIR / "data" / "phase5_verify_nb"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert (CAMPAIGN_DIR / "checkpoint.npz").exists(), \
    f"no campaign checkpoint at {CAMPAIGN_DIR} — has the Phase-4 campaign started + written to Drive?"
print("campaign:", CAMPAIGN_DIR, "| verify out:", OUT_DIR, "| top_k:", TOP_K, "workers:", N_WORKERS)


## 4. Run the 3D verification

Each design is meshed + solved in its own process. Re-run any time to re-verify the campaign's *current* top-k.


In [ ]:
import run_phase5_verify as script
from fanopt.cfd.mesh import VolumeMeshParams
from fanopt.cfd.phase5 import VerifyConfig

cfg = VerifyConfig(
    n_cycles=N_CYCLES, inner_iter=INNER_ITER,
    mesh_params=VolumeMeshParams(wall_mesh_size_m=WALL_MESH_SIZE_M),
)
summary = script.run(
    campaign_dir=CAMPAIGN_DIR, out_dir=OUT_DIR, top_k=TOP_K,
    su2_bin=None, cfg=cfg, n_workers=N_WORKERS,
)
print("verified", len(summary["designs"]), "designs | ranking:", summary["ranking"])
summary["designs"]


## 5. Slice vs 3D J_fan


In [ ]:
import json
import matplotlib.pyplot as plt

data = json.loads((OUT_DIR / "verification.json").read_text())
r = data["ranking"]
v = r["valid_only"]

# Full-spectrum: three rank/correlation metrics over the physically-valid designs,
# with suspect (negative or failed 3D J_fan) designs flagged rather than hidden.
print(f"valid n={v['n']}:  kendall τ={v['kendall_tau']}  spearman ρ={v['spearman_rho']}  "
      f"pearson R²={v['pearson_r2']}   rank_preserved={r['rank_preserved']}")
if r["n_suspect"]:
    print(f"⚠ {r['n_suspect']} suspect (negative/failed 3D J_fan): {r['suspect_designs']}")

pairs = [p for p in r["pairs"] if p["j_fan_slice"] is not None and p["j_fan_3d"] is not None]
fig, ax = plt.subplots(figsize=(5.8, 4.4))
for p in pairs:
    s = p["suspect"]
    ax.scatter(p["j_fan_slice"], p["j_fan_3d"], s=90,
               color="crimson" if s else "steelblue", marker="x" if s else "o")
    ax.annotate(p["name"] + (" (suspect)" if s else ""),
                (p["j_fan_slice"], p["j_fan_3d"]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
ax.axhline(0, color="gray", lw=0.8, ls="--")  # 3D J_fan < 0 = net reverse thrust
ax.set_xlabel("2D-slice J_fan  (cheap screening)")
ax.set_ylabel("3D J_fan  (high-fidelity verification)")
ax.set_title(f"Phase 5 — slice vs 3D | valid τ={v['kendall_tau']} ρ={v['spearman_rho']} "
             f"R²={v['pearson_r2']} | suspect={r['n_suspect']}")
fig.tight_layout()
plt.show()
